In [ ]:
///- --package ../dice

:< rust_operators
:< sm'_operators
:< digest

# dice_contract

In [ ]:
///- --test static

:< testing

## entropy

In [ ]:
:== entropy =
    {
        account_balance_high : u64
        account_balance_low : u64
        block_height : u64
        block_timestamp : u64
        epoch_height : u64
        key : string
        predecessor : string
        proof : string
        seed : array u8
        seeds : array u8
        signer : string
    }

:== call =
    {
        key : string
        proof : string
        seeds : array u8
    }

:= u64_text (x : u64) : string =
    #?x

:= part_bytes \* t. (x : t) : array u8 =
    ~>
        ?: t ?|
        | u64 => bytes.le_u64 x
        | string => bytes.utf8 x
        | array u8 => x

:= part_text \* t. (x : t) : string =
    ~>
        ?: t ?|
        | u64 => u64_text x
        | string => x
        | array u8 => bytes.hex x

:= entropy_bytes (e : entropy) : array u8 =
    bytes.concat (
        ~>
            real_core::record_type_fold_back
                (\ key => \* field. acc => (::) `(array u8) (part_bytes `field (e key)) acc)
                `entropy
                (Nil `(array u8))
    )

:= entropy_text (e : entropy) : string =
    ~>
        real_core::record_type_fold_back
            (\ key => \* field. acc =>
                (::) `(string * string) (sm'_real::symbol_to_string_real `(`key), part_text `field (e key)) acc
            )
            `entropy
            (Nil `(string * string))
    |> listm.fold (\ (first, acc) (name, value) => !-, acc ++# (? first ?> "" |: "; ") ++# name ++# " = " ++# value) (!+, "")
    |> snd

## chain

In [ ]:
:? read_entropy c : c -> call -> entropy

:? fail_with c : \* t. c -> string -> t

:! host = ()

:+ read_entropy host =
    \ _ (call : call) =>
        := low, high = near.host_account_balance_words ()
        {
            account_balance_high = high
            account_balance_low = low
            block_height = near.host_block_height ()
            block_timestamp = near.host_block_timestamp ()
            epoch_height = near.host_epoch_height ()
            key = call.key
            predecessor = near.host_predecessor ()
            proof = call.proof
            seed = near.host_random_seed ()
            seeds = call.seeds
            signer = near.host_signer ()
        }

:+ fail_with host =
    \ _ text => near.panic text

:+ digest_with host =
    \ _ digest => near.host_keccak512 digest

:! fixture =
    {
        account_balance : u64 * u64
        block_height : u64
        block_timestamp : u64
        epoch_height : u64
        predecessor : string
        seed : array u8
        signer : string
    }

:+ read_entropy fixture =
    \ (fixture f) (call : call) =>
        := low, high = f.account_balance
        {
            account_balance_high = high
            account_balance_low = low
            block_height = f.block_height
            block_timestamp = f.block_timestamp
            epoch_height = f.epoch_height
            key = call.key
            predecessor = f.predecessor
            proof = call.proof
            seed = f.seed
            seeds = call.seeds
            signer = f.signer
        }

:+ fail_with fixture =
    \ _ text => failwith text

:+ digest_with fixture =
    \ _ digest => keccak.keccak512 digest |> keccak.digest_bytes

:= sample_fixture () : fixture =
    fixture {
        account_balance = (7u64, 1u64)
        block_height = 42u64
        block_timestamp = 1700000000000000000u64
        epoch_height = 3u64
        predecessor = "alice.near"
        seed = bytes.repeated 32i32 7u8
        signer = "alice.near"
    }

:= sample_call (key : string) : call =
    { key = key; proof = ""; seeds = bytes.counting 4i32 }

#### entropy_bytes_follow_the_field_order_of_the_record (test)

In [1]:
///- --test
///> rust
///> fsharp
///> typescript
///> python
///> lua
///> c
///> zig
///> gleam
///> pascal

:= ~key = "k"
read_entropy (sample_fixture ()) (sample_call key)
|> entropy_bytes
|> keccak.keccak512
|> keccak.digest_bytes
|> bytes.hex
|> sm'.slice 0i32 15i32
|> _assert_eq "0e10e94fc9dfb460"

.rs (rust)
__assert_eq { expected = 0e10e94fc9dfb460 }
.fsx (fsharp)
__assert_eq { expected = 0e10e94fc9dfb460 }
.ts (typescript)
__assert_eq { expected = 0e10e94fc9dfb460 }
.py (python)
__assert_eq { expected = 0e10e94fc9dfb460 }
.lua (lua)
__assert_eq { expected = 0e10e94fc9dfb460 }
.c (c)
__assert_eq { expected = 0e10e94fc9dfb460 }
.zig (zig)
__assert_eq { expected = 0e10e94fc9dfb460 }
.gleam (gleam)
__assert_eq { expected = 0e10e94fc9dfb460 }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

#### entropy_text_names_every_field_once (test)

In [2]:
///- --test
///> rust
///> fsharp
///> typescript
///> python
///> lua
///> c
///> zig
///> gleam
///> pascal

:= ~key = "k"
read_entropy (sample_fixture ()) (sample_call key)
|> entropy_text
|> _assert_eq "account_balance_high = 1; account_balance_low = 7; block_height = 42; block_timestamp = 1700000000000000000; epoch_height = 3; key = k; predecessor = alice.near; proof = ; seed = 0707070707070707070707070707070707070707070707070707070707070707; seeds = 00010203; signer = alice.near"

.rs (rust)
__assert_eq { expected = account_balance_high = 1; account_balance_low = 7; block_height = 42; block_timestamp = 1700000000000000000; epoch_height = 3; key = k; predecessor = alice.near; proof = ; seed = 0707070707070707070707070707070707070707070707070707070707070707; seeds = 00010203; signer = alice.near }
.fsx (fsharp)
__assert_eq { expected = account_balance_high = 1; account_balance_low = 7; block_height = 42; block_timestamp = 1700000000000000000; epoch_height = 3; key = k; predecessor = alice.near; proof = ; seed = 0707070707070707070707070707070707070707070707070707070707070707; seeds = 00010203; signer = alice.near }
.ts (typescript)
__assert_eq { expected = account_balance_high = 1; account_balance_low = 7; block_height = 42; block_timestamp = 1700000000000000000; epoch_height = 3; key = k; predecessor = alice.near; proof = ; seed = 0707070707070707070707070707070707070707070707070707070707070707; seeds = 00010203; signer = alice.near }
.py (python)
__assert_eq { e

0

## phases

In [ ]:
:! requested = ()

:! bounded = ()

:! gathered = ()

:! hashed = ()

:! rolled = ()

:| roll_state phase =
    | Requested :: u64 * call -> roll_state requested
    | Bounded :: lib.dice.bound u64 * call -> roll_state bounded
    | Gathered :: lib.dice.bound u64 * entropy -> roll_state gathered
    | Hashed :: lib.dice.bound u64 * entropy * array u8 -> roll_state hashed
    | Rolled :: u64 * entropy * array u8 -> roll_state rolled

:| roll_step from to =
    | CheckBound :: roll_step requested bounded
    | Gather :: roll_step bounded gathered
    | Hash :: roll_step gathered hashed
    | Draw :: roll_step hashed rolled

:| ^ roll_plan from to =
    | Step :: \* from to. roll_step from to -> roll_plan from to
    | Then :: \* from to. (\? middle. roll_plan from middle * roll_plan middle to) -> roll_plan from to

:= check_bound \* c {fail_with}. (chain : c) (state : roll_state requested) : roll_state bounded =
    ?= state ?|
    | Requested (max, call) =>
        := b =
            ?= lib.dice.bound_of max ?|
            | Ok b => b
            | Error e => fail_with chain ("dice_contract.generate_random_number / " ++# lib.dice.bound_error_text e)
        Bounded (b, call)

:= gather \* c {read_entropy}. (chain : c) (state : roll_state bounded) : roll_state gathered =
    ?= state ?|
    | Bounded (b, call) => Gathered (b, read_entropy chain call)

:= hash \* c {digest_with}. (chain : c) (state : roll_state gathered) : roll_state hashed =
    ?= state ?|
    | Gathered (b, e) => Hashed (b, e, digest_with chain (entropy_bytes e))

:= draw \* c {digest_with}. (chain : c) (state : roll_state hashed) : roll_state rolled =
    ?= state ?|
    | Hashed (b, e, digest) =>
        := value, _ = lib.dice.roll lib.dice.Recycle b (lib.dice.hashed chain digest)
        Rolled (value, e, digest)

:= run_step \* from to c {read_entropy; fail_with; digest_with}.
    (chain : c)
    (step : roll_step from to)
    (state : roll_state from)
    : roll_state to
    =
    ?= step ?|
    | CheckBound => check_bound chain state
    | Gather => gather chain state
    | Hash => hash chain state
    | Draw => draw chain state

:= ^ run \* from to c {read_entropy; fail_with; digest_with}.
    (chain : c)
    (plan : roll_plan from to)
    (state : roll_state from)
    : roll_state to
    =
    ?= plan ?|
    | Step step => run_step chain step state
    | Then pair =>
        := \? middle. first, second = pair
        run chain second (run chain first state)

:= generate_plan () : roll_plan requested rolled =
    Then (\? (Step CheckBound, Then (\? (Step Gather, Then (\? (Step Hash, Step Draw))))))

:= generate \* c {read_entropy; fail_with; digest_with}. (chain : c) (call : call) (max : u64) : roll_state rolled =
    run chain (generate_plan ()) (Requested (max, call))

:= rolled_value (state : roll_state rolled) : u64 =
    ?= state ?|
    | Rolled (value, _, _) => value

#### a_fixture_rolls_the_same_number_on_every_backend (test)

In [3]:
///- --test
///> rust
///> fsharp
///> typescript
///> python
///> lua
///> c
///> zig
///> gleam
///> pascal

:= ~max = 2000u64
(generate (sample_fixture ()) (sample_call "k") max |> rolled_value, generate (sample_fixture ()) (sample_call "k2") max |> rolled_value)
|> _assert_eq (1646u64, 1834u64)

.rs (rust)
__assert_eq { expected = (1646, 1834) }
.fsx (fsharp)
__assert_eq { expected = (1646, 1834) }
.ts (typescript)
__assert_eq { expected = (1646, 1834) }
.py (python)
__assert_eq { expected = (1646, 1834) }
.lua (lua)
__assert_eq { expected = (1646, 1834) }
.c (c)
__assert_eq { expected = (1646, 1834) }
.zig (zig)
__assert_eq { expected = (1646, 1834) }
.gleam (gleam)
__assert_eq { expected = (1646, 1834) }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

#### the_bound_is_checked_before_anything_is_gathered (test)

In [4]:
///- --test
///> rust
///> fsharp
///> typescript
///> python

:= ~max = 0u64
\ () => generate (sample_fixture ()) (sample_call "k") max |> rolled_value |> ignore
|> _throws
|> optionm.map (sm'.format_exception >> sm'.contains "dice_contract.generate_random_number / max must be at least 1")
|> _assert_eq (Some !+)

.rs (rust)
__assert_eq { expected = Some(true) }
.fsx (fsharp)
__assert_eq { expected = Some(true) }
.ts (typescript)
__assert_eq { expected = Some(true) }
.py (python)
__assert_eq { expected = Some(true) }

0

#### a_plan_cannot_draw_before_hashing (test)

In [5]:
///- --test --fails ETY-TermError "Unification failure"
///> zig

:= plan : roll_plan requested rolled = Then (\? (Step CheckBound, Then (\? (Step Gather, Step Draw))))
0i32

failed to compile as expected (ETY-TermError "Unification failure"): FatalError: File main has a type error somewhere in its path. /home/runner/work/dice/dice/contract/dice_contract.spi:295:99 (main.spi:226:99): ETY-TermError: Unification failure. Got:      roll_step gathered rolled Expected: roll_step hashed rolled In the definition: spiral_cells_test.


#### a_plan_cannot_gather_before_the_bound_is_checked (test)

In [6]:
///- --test --fails ETY-TermError "Unification failure"
///> zig

:= plan : roll_plan requested rolled = Then (\? (Step Gather, Then (\? (Step CheckBound, Then (\? (Step Hash, Step Draw))))))
0i32

failed to compile as expected (ETY-TermError "Unification failure"): FatalError: File main has a type error somewhere in its path. /home/runner/work/dice/dice/contract/dice_contract.spi:300:59 (main.spi:226:59): ETY-TermError: Unification failure. Got:      roll_step requested gathered Expected: roll_step bounded gathered In the definition: spiral_cells_test. /home/runner/work/dice/dice/contract/dice_contract.spi:300:82 (main.spi:226:82): ETY-TermError: Unification failure. Got:      roll_step gathered bounded Expected: roll_step requested bounded In the definition: spiral_cells_test. /home/runner/work/dice/dice/contract/dice_contract.spi:300:109 (main.spi:226:109): ETY-TermError: Unification failure. Got:      roll_step bounded hashed Expected: roll_step gathered hashed In the definition: spiral_cells_test.


#### a_rolled_state_cannot_be_rolled_again (test)

In [7]:
///- --test --fails ETY-TermError "Unification failure"
///> zig

:= again (state : roll_state rolled) : roll_state rolled = draw (sample_fixture ()) state
0i32

failed to compile as expected (ETY-TermError "Unification failure"): FatalError: File main has a type error somewhere in its path. /home/runner/work/dice/dice/contract/dice_contract.spi:305:89 (main.spi:226:89): ETY-TermError: Unification failure. Got:      roll_state hashed Expected: roll_state rolled In the definition: spiral_cells_test.


## seeds

In [ ]:
:= new () : u32 * am'.vec u8 =
    3u32, (([] : list u8) |> am'.new_vec)

:= max_seeds () : convert.unativeint =
    100i32 |> convert.convert

:= trim_seeds (seeds : rust.ref (rust.mut' (am'.vec u8))) : am'.vec u8 =
    := len : u64 = seeds |> am'.vec_ref_len |> convert.convert
    := max : u64 = max_seeds () |> convert.convert
    := excess = ? len > max ?> len - max |: 0u64
    seeds |> am'.vec_drain_front (excess |> convert.convert)

:= contribute_seed (seeds : rust.ref (rust.mut' (am'.vec u8))) (seed : am'.vec u8) : () =
    := seed_len = seed |> am'.vec_len
    seeds |> am'.vec_extend_from_slice seed
    := seed_excess = seeds |> trim_seeds
    trace Debug
        \ () => "dice_contract.contribute_seed"
        \ () => { seed_len seed_excess = seed_excess |> crypto.hex_of_vec }
    trace_flush ()

#### contribute_seed_keeps_the_newest_hundred_bytes (test)

In [8]:
///- --test
///> rust

:= seeds = ([] : list u8) |> am'.new_vec |> rust.to_ref_mut
listm'.init_series 1i32 120 1 |> listm.map convert.u8 |> am'.new_vec |> contribute_seed seeds
((seeds |> rust.len : u32), (seeds |> am'.vec_ref_len |> convert.convert : u64))
|> _assert_eq (100u32, 100u64)

21:58:19 1d dice_contract.contribute_seed / { seed_len = 120; seed_excess = 0102030405060708090a0b0c0d0e0f1011121314 }

__assert_eq { expected = (100, 100) }

0

## generate_random_number

In [ ]:
:= generate_random_number
    (seeds : rust.ref (rust.mut' (am'.vec u8)))
    (key : sm'.std_string)
    (proof : sm'.std_string)
    (max : u64)
    : u64
    =
    := call = { key = key |> sm'.from_std_string; proof = proof |> sm'.from_std_string; seeds = seeds |> am'.vec_ref_to_array }
    ?= generate host call max ?|
    | Rolled (value, e, digest) =>
        seeds |> am'.vec_extend_from_array digest
        seeds |> trim_seeds |> ignore
        trace Debug
            \ () => "dice_contract.generate_random_number " ++# entropy_text e ++# "; hash = " ++# bytes.hex digest
            \ () => { max result = value }
        trace_flush ()
        value

:= roll_within_bounds (max : u64) (rolls : am'.vec u8) : optionm'.option' u64 =
    := result =
        rolls
        |> am'.from_vec_base
        |> am'.to_list_base'
        |> listm'.unbox
        |> lib.dice.roll_within_bounds max
        |> optionm'.box
    trace Debug
        \ () => "dice_contract.roll_within_bounds"
        \ () => { max rolls = #?*rolls; result = #?*result }
    trace_flush ()
    result

## host

In [ ]:
:= mocked_chain () : () =
    $'near_sdk::testing_env!!(near_sdk::test_utils::VMContextBuilder::new().random_seed([7u8; 32]).block_height(42).block_timestamp(1_700_000_000_000_000_000).epoch_height(3).account_balance(near_sdk::NearToken::from_yoctonear((1u128 << 64) + 7)).signer_account_id("alice.near".parse().unwrap()).predecessor_account_id("alice.near".parse().unwrap()).build())' : ()

#### the_host_reads_the_chain_the_fixture_describes (test)

In [9]:
///- --test
///> rust -d "near-sdk={version='=5.30.1',features=['unit-testing']}" -d near-token

mocked_chain ()
:= host_entropy = read_entropy host (sample_call "k")
:= fixture_entropy = read_entropy (sample_fixture ()) (sample_call "k")
(entropy_text host_entropy = entropy_text fixture_entropy, entropy_bytes host_entropy |> bytes.hex = (entropy_bytes fixture_entropy |> bytes.hex))
|> _assert_eq (!+, !+)

__assert_eq { expected = (true, true) }

#### the_host_rolls_what_the_fixture_rolls (test)

In [10]:
///- --test
///> rust -d "near-sdk={version='=5.30.1',features=['unit-testing']}" -d near-token

mocked_chain ()
(generate host (sample_call "k") 2000u64 |> rolled_value, generate (sample_fixture ()) (sample_call "k") 2000u64 |> rolled_value)
|> _assert_eq (1646u64, 1646u64)

__assert_eq { expected = (1646, 1646) }

#### the_host_keccak_is_the_spiral_keccak (test)

In [11]:
///- --test
///> rust -d "near-sdk={version='=5.30.1',features=['unit-testing']}" -d near-token

mocked_chain ()
:= ~size = 150i32
:= message = bytes.counting size
near.host_keccak512 message |> bytes.hex
|> _assert_eq (keccak.keccak512 message |> keccak.digest_bytes |> bytes.hex)

__assert_eq { expected = 7bcdbadf9edc379a1de2a1421817b825120112e6f46fdf30a838d423fee413c5f8cedd3730bf140c78cb83b9bb411690ad70ddf288d73858f9408338f1845595 }

#### generate_random_number_appends_the_hash_to_the_seeds (test)

In [12]:
///- --test
///> rust -d "near-sdk={version='=5.30.1',features=['unit-testing']}" -d near-token

mocked_chain ()
:= seeds = ([] : list u8) |> am'.new_vec |> rust.to_ref_mut
:= key = "k" |> sm'.to_std_string
:= proof = "" |> sm'.to_std_string
:= first = generate_random_number seeds key proof 2000u64
:= after_first : u64 = seeds |> am'.vec_ref_len |> convert.convert
:= second = generate_random_number seeds key proof 2000u64
:= after_second : u64 = seeds |> am'.vec_ref_len |> convert.convert
(first, after_first, second, after_second)
|> _assert_eq (762u64, 64u64, 1614u64, 100u64)

21:59:44 1d dice_contract.generate_random_number account_balance_high = 1; account_balance_low = 7; block_height = 42; block_timestamp = 1700000000000000000; epoch_height = 3; key = k; predecessor = alice.near; proof = ; seed = 0707070707070707070707070707070707070707070707070707070707070707; seeds = ; signer = alice.near; hash = e0d870d6e57fe3f90875bc51c47c58a2e39bbd51ca92fe26d145d57f9ef1eedde73122bd024c7640ef39e50b530f29f164680d0a9ed0d6a484413ec56c008b45 / { max = 2000; result = 762 }

21:59:44 3d dice_contract.generate_random_number account_balance_high = 1; account_balance_low = 7; block_height = 42; block_timestamp = 1700000000000000000; epoch_height = 3; key = k; predecessor = alice.near; proof = ; seed = 0707070707070707070707070707070707070707070707070707070707070707; seeds = e0d870d6e57fe3f90875bc51c47c58a2e39bbd51ca92fe26d145d57f9ef1eedde73122bd024c7640ef39e50b530f29f164680d0a9ed0d6a484413ec56c008b45; signer = alice.near; hash = 29fe6cd5b668f7176ce4ca33a04e96dc322bb1503c7bb29

#### the_host_glue_pins_its_gas (test)

In [13]:
///- --test
///> rust -d "near-sdk={version='=5.30.1',features=['unit-testing']}" -d near-token

mocked_chain ()
:= used () : u64 = $'near_sdk::env::used_gas().as_gas()' : u64
:= start = used ()
:= seed = near.host_random_seed ()
:= after_seed = used ()
:= digest = near.host_keccak512 (bytes.counting 150i32)
:= after_hash = used ()
:= state = generate host (sample_call "k") 2000u64
:= after_roll = used ()
(after_seed - start, after_hash - after_seed, after_roll - after_hash, length seed + length digest, rolled_value state)
|> _assert_eq (9192751602u64, 23628854763u64, 52413432801u64, 96i32, 1646u64)

__assert_eq { expected = (9192751602, 23628854763, 52413432801, 96, 1646) }

### main

In [ ]:
:= quiet () : () =
    set_trace_level Debug

:= state_struct () : () =
    global """
        #[near_sdk::near(contract_state)]
        #[derive(near_sdk::PanicOnDefault)]
        pub struct State((u32, Vec<u8>));
        """

:= legacy_state () : () =
    global """
        #[derive(near_sdk::PanicOnDefault, borsh::BorshDeserialize, borsh::BorshSerialize)]
        pub struct OldState {
            version: u32,
            seeds: near_sdk::store::vec::Vector<u8>,
        }
        """
    global """
        impl From<OldState> for State {
            fn from(mut old_state: OldState) -> Self {
                let seeds = old_state.seeds.iter().copied().collect();
                old_state.seeds.clear();
                old_state.seeds.flush();
                Self((old_state.version + 1, seeds))
            }
        }
        """

:= init_methods () : () =
    global """
        #[near_sdk::near]
        impl State {
            #[init]
            pub fn new() -> Self {
                Self(dice_contract_new())
            }
            #[private]
            #[init(ignore_state)]
            pub fn migrate() -> Self {
                let raw = near_sdk::env::storage_read(b"STATE")
                    .unwrap_or_else(|| near_sdk::env::panic_str("dice_contract.migrate / no state to migrate"));
                if raw.len() >= 4 && u32::from_le_bytes([raw[0], raw[1], raw[2], raw[3]]) >= 3 {
                    near_sdk::env::panic_str("dice_contract.migrate / the state is already migrated")
                }
                borsh::from_slice::<OldState>(&raw)
                    .unwrap_or_else(|_| near_sdk::env::panic_str("dice_contract.migrate / the state is not the old layout"))
                    .into()
            }
        }
        """
    export
        "dice_contract_new"
        ((\ () => := () = quiet () ;; new ()) : () -> u32 * am'.vec u8)

:= seed_methods () : () =
    global """
        #[near_sdk::near]
        impl State {
            pub fn contribute_seed(&mut self, seed: Vec<u8>) {
                dice_contract_contribute_seed(&mut self.0.1, seed)
            }
            pub fn contribute_seed_borsh(&mut self, #[serializer(borsh)] seed: Vec<u8>) {
                self.contribute_seed(seed)
            }
        }
        """
    export
        "dice_contract_contribute_seed"
        (
            \ (seeds, seed) =>
                := () = quiet ()
                contribute_seed seeds seed
            : rust.ref (rust.mut' (am'.vec u8)) * am'.vec u8 -> ()
        )

:= roll_methods () : () =
    global """
        #[near_sdk::near]
        impl State {
            pub fn generate_random_number(&mut self, key: String, proof: String, max: u64) -> u64 {
                dice_contract_generate_random_number(&mut self.0.1, key, proof, max)
            }
        }
        """
    export
        "dice_contract_generate_random_number"
        (
            \ (seeds, key, proof, max) =>
                := () = quiet ()
                generate_random_number seeds key proof max
            : rust.ref (rust.mut' (am'.vec u8)) * sm'.std_string * sm'.std_string * u64 -> u64
        )

:= bound_methods () : () =
    global """
        #[near_sdk::near]
        impl State {
            pub fn roll_within_bounds(max: u64, rolls: Vec<u8>) -> Option<u64> {
                dice_contract_roll_within_bounds(max, rolls)
            }
            #[result_serializer(borsh)]
            pub fn roll_within_bounds_borsh(
                #[serializer(borsh)] max: u64,
                #[serializer(borsh)] rolls: Vec<u8>,
            ) -> Option<u64> {
                Self::roll_within_bounds(max, rolls)
            }
        }
        """
    export
        "dice_contract_roll_within_bounds"
        (
            \ (max, rolls) =>
                := () = quiet ()
                roll_within_bounds max rolls
            : u64 * am'.vec u8 -> optionm'.option' u64
        )

:= main_native () : () =
    near.bump_allocator ()
    state_struct ()
    legacy_state ()
    init_methods ()
    seed_methods ()
    roll_methods ()
    bound_methods ()

:= main () =
    ? is_rust ()
    ?> main_native ()
    |: ()